# **LAB 5**

- **NAME**: *ABDUL HADI*
- **REG #**: *SP24BDS001*

In [1]:
import nltk
from nltk.util import ngrams
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline
import pandas as pd
import numpy as np

# Download required nltk data if needed
nltk.download('punkt')

# Sample product reviews corpus (simulating 'product_reviews_corpus')
corpus_text = [
    "the camera quality exceeded my expectations",
    "battery life is terrible and the screen is dim",
    "great product for the price highly recommend",
    "exceeded my expectations in every way possible",
    "the battery drains too fast during video playback",
    "superb camera quality and crystal clear screen"
]

# Tokenize corpus
train_data = [nltk.word_tokenize(sent.lower()) for sent in corpus_text]

# Held-out sentence
held_out_sentence = "the camera quality exceeded my expectations"
test_data = [nltk.word_tokenize(held_out_sentence.lower())]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hadi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### **Activity 1**

In [6]:
# Prepare n-grams for bigrams (order = 2)
n = 2
train_igrams, vocab = padded_everygram_pipeline(n, train_data)
test_igrams, _ = padded_everygram_pipeline(n, test_data)

# Flatten test grams for evaluation
test_igrams_list = list(test_igrams)

# 1. Laplace (Add-1) Smoothing
laplace_model = Laplace(n)
laplace_model.fit(train_igrams, vocab)

# Recreate train grams since generator is consumed
train_igrams_1, vocab_1 = padded_everygram_pipeline(n, train_data)
# 2. Kneser-Ney Smoothing
kn_model = KneserNeyInterpolated(n)
kn_model.fit(train_igrams_1, vocab_1)

# Compute Perplexity
laplace_perplexity = laplace_model.perplexity(test_data[0])
kn_perplexity = kn_model.perplexity(test_data[0])

print(f"Laplace Bigram Perplexity: {laplace_perplexity:.4f}")
print(f"Kneser-Ney Bigram Perplexity: {kn_perplexity:.4f}")

Laplace Bigram Perplexity: 35.0000
Kneser-Ney Bigram Perplexity: inf


Conclusion: Laplace performed better because it achieved a lower perplexity score, indicating a better probabilistic fit for the unseen sentence.

### **Activity 2**


In [7]:
# Best model from Activity 1 (Kneser-Ney)
candidates = ["expectations", "budget", "screen"]
context = ("camera", "quality", "exceeded", "my") # Last n-1 words for bigram/trigram context

# Score candidates using conditional probability P(word | context)
scores = {}
for word in candidates:
    # For KneserNey or trained model score lookup
    score = kn_model.score(word, context[-1:]) # Using bigram context (last word 'my')
    scores[word] = score

ranked_candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("Ranked Probabilities for Next Word:")
for word, prob in ranked_candidates:
    print(f"- {word}: {prob:.6f}")

print(f"\nSuggested First Word: '{ranked_candidates[0][0]}'")

Ranked Probabilities for Next Word:
- expectations: 0.951087
- screen: 0.002174
- budget: 0.000000

Suggested First Word: 'expectations'


### **Activity 3**

In [8]:
# Bigram Model
train_bi, vocab_bi = padded_everygram_pipeline(2, train_data)
bi_model = KneserNeyInterpolated(2)
bi_model.fit(train_bi, vocab_bi)

# Trigram Model
train_tri, vocab_tri = padded_everygram_pipeline(3, train_data)
tri_model = KneserNeyInterpolated(3)
tri_model.fit(train_tri, vocab_tri)

bi_perp = bi_model.perplexity(test_data[0])
tri_perp = tri_model.perplexity(test_data[0])

print(f"Bigram Perplexity: {bi_perp:.4f}")
print(f"Trigram Perplexity: {tri_perp:.4f}")

Bigram Perplexity: inf
Trigram Perplexity: inf


Explanation: The trigram model captures longer context but can suffer from sparsity if the exact 3-word sequence wasn't frequent in training, whereas the bigram model generalizes more broadly.

### **Activity 4**

In [9]:
oov_sentence = "the camera quality exceeded my expectations extraordinarily"
oov_test_data = [nltk.word_tokenize(oov_sentence.lower())]

# Train un-smoothed MLE model
train_mle, vocab_mle = padded_everygram_pipeline(2, train_data)
mle_model = MLE(2)
mle_model.fit(train_mle, vocab_mle)

# Compute perplexity (MLE will output inf or division by zero due to zero probability of unseen words)
try:
    mle_perp = mle_model.perplexity(oov_test_data[0])
except ZeroDivisionError:
    mle_perp = float('inf')

smoothed_oov_perp = kn_model.perplexity(oov_test_data[0])

print(f"Unsmoothed (MLE) Model Perplexity with OOV: {mle_perp}")
print(f"Smoothed (Kneser-Ney) Model Perplexity with OOV: {smoothed_oov_perp:.4f}")

Unsmoothed (MLE) Model Perplexity with OOV: inf
Smoothed (Kneser-Ney) Model Perplexity with OOV: inf
